# Ling-Spam EDA

In [ ]:
import pandas as pd
import numpy as np
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

In [ ]:
from typing import Optional

def generate_word_cloud(
    body: pd.Series,
    stopwords: Optional[set[str]] = STOPWORDS,
    width: int = 800,
    height: int = 800,
    background_color: str = "white",
    min_font_size: int = 10
) -> WordCloud:
    r"""
    Function that takes the bodies of the emails as a parameter and return a word cloud.
    
    Parameters
    ----------
    body (pd.Series): A pandas Series that contains the bodies of the emails
    stopwords (Optional[set[str]]): An optional set of string that contains the stopwords
    width (int): The width of the generated word cloud image
    height (int): The height of the generated word cloud images
    background_color (str): The background color of the generated word cloud image
    min_font_sizee (int): The minimal font size in the generated word cloud image
    
    Returns
    -------
    WordCloud
    """
    
    if stopwords is None:
        stopwords = set()
    words = ""

    # iterate through the csv file
    for val in body:

        # typecaste each val to string
        val = str(val)

        # split the value
        tokens = val.split()

        # Converts each token into lowercase
        for i in range(len(tokens)):
            tokens[i] = tokens[i].lower()

        words += " ".join(tokens)+" "

    wordcloud = WordCloud(width=width, height=height,
                    background_color=background_color,
                    stopwords=stopwords,
                    min_font_size=min_font_size).generate(words)
    return wordcloud

In [ ]:
ling_spam_df = pd.read_csv("/kaggle/input/email-spam-dataset/lingSpam.csv")
print(f"Ling Spam dataset contains {ling_spam_df.shape[0]} rows and {ling_spam_df.shape[1]} columns")
print(f"Columns: {list(ling_spam_df.columns)}")
ling_spam_df.head()

In [ ]:
# Drop "Unnamed: 0" and "Unnamed: 0.1" columns
ls_df = ling_spam_df.copy()
ls_df.drop(["Unnamed: 0"], axis=1, inplace=True)

In [ ]:
print(f"Ling Spam dataset contains {ls_df.isna().sum().sum()} missing values")
ls_df.isna().sum()

In [ ]:
plt.title("Target Distribution")
ls_df["Label"].map({0: "ham", 1: "spam"}).value_counts().plot.pie(autopct='%1.1f%%', shadow=True)
plt.show()

In [ ]:
num_examples_per_category = 10

positive_df = ls_df[ls_df["Label"] == 1]
negative_df = ls_df[ls_df["Label"] == 0]

print(f"There are {positive_df.shape[0]} positive samples and {negative_df.shape[0]} negative samples.\n\n")

for index, body in enumerate(positive_df["Body"].values[:num_examples_per_category], 1):
    print(f"Positive examples {index}: {body}")
    print("-"*80+"\n")
    
print("="*80+"\n")
for index, body in enumerate(negative_df["Body"].values[:num_examples_per_category], 1):
    print(f"Negative examples {index}: {body}")

In [ ]:
wordcloud = generate_word_cloud(ls_df["Body"])

# plot the WordCloud image
plt.figure(figsize=(8, 8), facecolor=None)
plt.title("Ling Spam Dataset Word Cloud")
plt.imshow(wordcloud)
plt.axis("off")
plt.tight_layout(pad=0)
 
plt.show()

In [ ]:
positive_wordcloud = generate_word_cloud(positive_df["Body"])
negative_wordcloud = generate_word_cloud(negative_df["Body"])

# plot the WordCloud image
fig, axes = plt.subplots(1, 2, figsize=(14, 8), facecolor=None)
axes[0].set_title("Negative Samples Word Cloud")
axes[0].imshow(negative_wordcloud)
axes[1].set_title("Positive Samples Word Cloud")
axes[1].imshow(positive_wordcloud)
plt.axis("off")
plt.tight_layout(pad=0)
 
plt.show()

In [ ]:
def get_n_words(body: str) -> int:
    """
    A function that takes a body (string) as parameter and returns the number of words in the body.
    """
    return len(body.split())

ls_df["n_words"] = ls_df["Body"].map(get_n_words)
ls_df.head()

In [ ]:
plt.figure(figsize=(14, 6))
plt.title("Number of words distribution")
sns.histplot(data=ls_df[ls_df["n_words"] < ls_df["n_words"].quantile(0.9999)], x="n_words", hue="Label")
plt.show()

In [ ]:
ls_df.groupby("Label").describe()